# Analyse de Performance - Vendeurs et Produits (Olist E-Commerce)

Ce notebook présente l'analyse complète de la performance des vendeurs et des produits de la marketplace Olist, de la phase exploratoire (EDA) jusqu'à la recommandation stratégique pour le comité commercial.

## Prémier pas

### Importation des librairies necessaires

In [22]:
# --- Bibliothèques standard ---
import io
import json
import os
import time
import zipfile
from pathlib import Path

# --- Bibliothèques tierces ---
import kagglehub
import numpy as np
import pandas as pd
import requests
from kagglehub import KaggleDatasetAdapter

### Chargement des données 

#### Méthode 1 : Chargement local par fichier CSV 

In [23]:
# # Chemin d'accès des fichiers dans le disque local
# DOSSIER_DONNEES = Path("C:/Users/Wagner/datasets/olistCommerce")

# # Dictionnaire d'association : noms logiques -> noms réels des fichiers CSV
# FICHIERS = {
#     "orders": "olist_orders_dataset.csv",
#     "items": "olist_order_items_dataset.csv",
#     "products": "olist_products_dataset.csv",
#     "cat_trans": "product_category_name_translation.csv",
#     "sellers": "olist_sellers_dataset.csv",
#     "customers": "olist_customers_dataset.csv",
#     "reviews": "olist_order_reviews_dataset.csv",
# }

# # Chargement direct de tous les fichiers CSV dans un dictionnaire
# tables = {
#     nom: pd.read_csv(DOSSIER_DONNEES / fichier, low_memory=False)
#     for nom, fichier in FICHIERS.items()
# }

# # Affichage d'un résumé rapide du chargement
# print("\nDATASET OLIST — CHARGEMENT LOCAL")
# print("-" * 45)
# for nom, df in tables.items():
#     print(f"{nom:<12} : {len(df):>7,} lignes | {len(df.columns):>2} colonnes")
# print("-" * 45)

# # Création des variables DataFrames individuelles pour un accès rapide
# df_orders = tables["orders"]
# df_products = tables["items"]
# df_products = tables["products"]
# df_cat_trans = tables["cat_trans"]
# df_sellers = tables["sellers"]
# df_customers = tables["customers"]
# df_reviews = tables["reviews"]

#### Méthode 2 : Chargement via Kaggle API

In [24]:
# Identifiant du dataset officiel sur Kaggle
DATASET = "olistbr/brazilian-ecommerce"

# Dictionnaire d'association : nom abrégé -> nom réel du fichier CSV
FICHIERS = {
    "orders": "olist_orders_dataset.csv",
    "items": "olist_order_items_dataset.csv",
    "products": "olist_products_dataset.csv",
    "cat_trans": "product_category_name_translation.csv",
    "sellers": "olist_sellers_dataset.csv",
    "customers": "olist_customers_dataset.csv",
    "reviews": "olist_order_reviews_dataset.csv",
}

# Récupération des identifiants Kaggle (dans le fichier local ~/.kaggle/kaggle.json ou les variables d'environnement)
k_file = Path.home() / ".kaggle" / "kaggle.json"
k_auth = json.loads(k_file.read_text()) if k_file.exists() else {}
user = os.getenv("KAGGLE_USERNAME") or k_auth.get("username")
key = os.getenv("KAGGLE_KEY") or k_auth.get("key")

# Vérification de la présence des identifiants API
if not (user and key):
    raise EnvironmentError("Identifiants Kaggle manquants.")

# Téléchargement du fichier ZIP de l'archive via l'API Kaggle
url = f"https://www.kaggle.com/api/v1/datasets/download/{DATASET}"
reponse = requests.get(url, auth=(user, key), timeout=(30, 120))
reponse.raise_for_status()

# Extraction et lecture directe des fichiers CSV en mémoire dans un dictionnaire de DataFrames
with zipfile.ZipFile(io.BytesIO(reponse.content)) as archive:
    fichiers = archive.namelist()
    tables = {
        nom: pd.read_csv(
            archive.open(next(f for f in fichiers if f.endswith(fichier))),
            low_memory=False,
        )
        for nom, fichier in FICHIERS.items()
    }

# Affichage de l'en-tête du tableau récapitulatif
print("\n" + "─" * 62)
print(f"  DATASET OLIST — {len(tables)} tables chargées")
print("─" * 62)
print(f"  {'Nom de la table':<24} {'Lignes':>10}   {'Colonnes':>8}   {'Mémoire':>10}")
print("─" * 62)

# Calcul et affichage des statistiques (lignes, colonnes, mémoire RAM) pour chaque DataFrame
tot_lignes, tot_memoire = 0, 0
for nom, df in tables.items():
    lignes, colonnes = len(df), len(df.columns)
    mem_mo = df.memory_usage(deep=True).sum() / 1024**2
    tot_lignes += lignes
    tot_memoire += mem_mo
    print(f"  {nom:<24} {lignes:>10,}   {colonnes:>8}   {mem_mo:>7.2f} Mo")

# Affichage du total
print("─" * 62)
print(f"  {'TOTAL':<24} {tot_lignes:>10,}   {'—':>8}   {tot_memoire:>7.2f} Mo")
print("─" * 62 + "\n")

# Affectation directe de chaque table du dictionnaire dans sa propre variable
df_orders = tables["orders"]
df_items = tables["items"]
df_products = tables["products"]
df_cat_trans = tables["cat_trans"]
df_sellers = tables["sellers"]
df_customers = tables["customers"]
df_reviews = tables["reviews"]


──────────────────────────────────────────────────────────────
  DATASET OLIST — 7 tables chargées
──────────────────────────────────────────────────────────────
  Nom de la table              Lignes   Colonnes      Mémoire
──────────────────────────────────────────────────────────────
  orders                       99,441          8     58.97 Mo
  items                       112,650          7     39.43 Mo
  products                     32,951          9      6.79 Mo
  cat_trans                        71          2      0.01 Mo
  sellers                       3,095          4      0.66 Mo
  customers                    99,441          5     29.62 Mo
  reviews                      99,224          7     42.75 Mo
──────────────────────────────────────────────────────────────
  TOTAL                       446,873          —    178.23 Mo
──────────────────────────────────────────────────────────────



## Partie EDA : Explorer avant de croiser

### Exercice 1 - Audit systématique des 7 tables

In [25]:
# Génération des métadonnées sous forme de liste de dictionnaires
donnees_audit = []

for nom, dataframe in tables.items():
    # Formatage de chaque colonne sous la forme "nom_colonne (type)"
    types_colonnes = ", ".join(
        [
            f"{colonne} ({type_donnee})"
            for colonne, type_donnee in dataframe.dtypes.items()
        ]
    )

    donnees_audit.append(
        {
            "Table": nom,
            "Lignes": f"{len(dataframe):,}",
            "Colonnes": len(dataframe.columns),
            "Types de donnée": types_colonnes,
        }
    )

# Création du DataFrame récapitulatif
dataframe_audit = pd.DataFrame(donnees_audit)

display(dataframe_audit)

,Table,Lignes,Colonnes,Types de donnée
0,orders,"99,441",8,"order_id (str), customer_id (str), order_statu..."
1,items,"112,650",7,"order_id (str), order_item_id (int64), product..."
2,products,"32,951",9,"product_id (str), product_category_name (str),..."
3,cat_trans,71,2,"product_category_name (str), product_category_..."
4,sellers,"3,095",4,"seller_id (str), seller_zip_code_prefix (int64..."
5,customers,"99,441",5,"customer_id (str), customer_unique_id (str), c..."
6,reviews,"99,224",7,"review_id (str), order_id (str), review_score ..."


### Exercice 2 - Valeurs manquantes : où et combien ?

In [26]:
donnees_valeurs_manquantes = []

for nom_table, dataframe in tables.items():
    lignes_totales = len(dataframe)
    # Compte du nombre de valeurs manquantes par colonne
    serie_manquants = dataframe.isna().sum()

    # Filtrage sur les colonnes qui possèdent au moins une valeur manquante
    cols_avec_manquants = serie_manquants[serie_manquants > 0]

    if cols_avec_manquants.empty:
        donnees_valeurs_manquantes.append(
            {
                "Table": nom_table,
                "Colonne": "Aucune",
                "Nombre de NaN": 0,
                "Proportion (%)": "0.00 %",
            }
        )
    else:
        for colonne, nb_nan in cols_avec_manquants.items():
            donnees_valeurs_manquantes.append(
                {
                    "Table": nom_table,
                    "Colonne": colonne,
                    "Nombre de NaN": f"{nb_nan:,}",
                    "Proportion (%)": f"{(nb_nan / lignes_totales) * 100:.2f} %",
                }
            )

dataframe_valeurs_manquantes = pd.DataFrame(donnees_valeurs_manquantes)

# Affichage avec mise en page
pd.set_option("display.max_rows", None)
display(dataframe_valeurs_manquantes)

,Table,Colonne,Nombre de NaN,Proportion (%)
0,orders,order_approved_at,160,0.16 %
1,orders,order_delivered_carrier_date,"1,783",1.79 %
2,orders,order_delivered_customer_date,"2,965",2.98 %
3,items,Aucune,0,0.00 %
4,products,product_category_name,610,1.85 %
5,products,product_name_lenght,610,1.85 %
6,products,product_description_lenght,610,1.85 %
7,products,product_photos_qty,610,1.85 %
8,products,product_weight_g,2,0.01 %
9,products,product_length_cm,2,0.01 %


#### Hypothèse métier

##### 1. Table `reviews` : Justification des commentaires absents

Dans la table `reviews`, la colonne `review_comment_message` comporte plus de **58 % de valeurs manquantes** (`NaN`), et la colonne `review_comment_title` près de **88 %**.

**Explication métier :**  
Il ne s'agit **pas d'un défaut du fichier** ni d'une erreur d'extraction. Lors de la notation d'une commande sur la plateforme Olist, l'attribution d'un score de 1 à 5 étoiles (`review_score`) est la seule étape obligatoire. La rédaction d'un titre et d'un commentaire reste entièrement facultative. La majorité des acheteurs (notamment lorsqu'ils sont satisfaits) attribuent une note sans rédiger de texte, ce qui explique naturellement l'absence de données dans ces colonnes.

---

##### 2. Table `orders` : Hypothèse sur l'absence des dates de livraison

Les colonnes `order_approved_at`, `order_delivered_carrier_date` et `order_delivered_customer_date` présentent plusieurs milliers de valeurs manquantes.

**Hypothèse métier :**  
L'absence de ces horodatages est **légitime** et s'explique par le statut et l'avancement de la commande dans le cycle de vie logistique (`order_status`) au moment de l'extraction des données :

* **`order_delivered_customer_date` absente** : La commande n'a pas encore été réceptionnée par le client. Elle est soit en cours de préparation (`invoiced`, `processing`), en cours d'acheminement (`shipped`), ou définitivement interrompue (`canceled`, `unavailable`).
* **`order_delivered_carrier_date` absente** : Le colis n'a pas encore été transmis au partenaire logistique par le vendeur.
* **`order_approved_at` absente** : Le paiement n'a pas encore été validé par l'organisme financier (par exemple lors de l'attente du règlement d'un *Boleto Bancário* ou en cas de refus de carte de crédit).

Ces manquements ne doivent donc pas être supprimés aveuglément : ils reflètent l'état opérationnel réel des flux logistiques d'Olist.

### Exercice 3 - Une clé qui ne tient pas sa promesse

In [43]:
# Statistiques
total = len(df_reviews)
uniques = df_reviews['review_id'].nunique()
doublons = total - uniques

print('\n' + '─' * 60)
print(' RAPPORT D\'AUDIT : UNICITÉ DE REVIEW_ID')
print('─' * 60)
print(f'Nombre total de lignes : {total:,}')
print(f'Nombre de review_id uniques : {uniques:,}')
print(f'Nombre de review_id en double : {doublons:,}')
print('─' * 60)

# Affichage des review_id en double
if doublons > 0:

    doublons_df = (
        df_reviews[
            df_reviews.duplicated(
                subset=["review_id"],
                keep=False
            )
        ][["review_id", "order_id"]]
        .sort_values("review_id")
    )

    print("─" * 60)
    print("DÉTAIL DES REVIEW_ID EN DOUBLE")
    print("─" * 60)
    print(f"Nombre de lignes concernées : {len(doublons_df):,}")
    print(f"Nombre de review_id concernés : {doublons_df['review_id'].nunique():,}")

    print("\nTABLEAU DES DOUBLONS")
    print("─" * 60)

    display(doublons_df)

    print("OBSERVATION")
    print("─" * 60)

    nb_commandes = doublons_df.groupby("review_id")["order_id"].nunique()

    if (nb_commandes > 1).any():
        print(
            "Certains review_id sont associés à plusieurs order_id."
        )
        print(
            "Cela peut indiquer qu'un même avis est associé à plusieurs commandes."
        )
    else:
        print(
            "Les review_id en double sont associés à une même commande."
        )



────────────────────────────────────────────────────────────
 RAPPORT D'AUDIT : UNICITÉ DE REVIEW_ID
────────────────────────────────────────────────────────────
Nombre total de lignes : 99,224
Nombre de review_id uniques : 98,410
Nombre de review_id en double : 814
────────────────────────────────────────────────────────────
────────────────────────────────────────────────────────────
DÉTAIL DES REVIEW_ID EN DOUBLE
────────────────────────────────────────────────────────────
Nombre de lignes concernées : 1,603
Nombre de review_id concernés : 789

TABLEAU DES DOUBLONS
────────────────────────────────────────────────────────────


,review_id,order_id
46678,00130cbe1f9d422698c812ed8ded1919,dfcdfc43867d1c1381bfaf62d6b9c195
29841,00130cbe1f9d422698c812ed8ded1919,04a28263e085d399c97ae49e0b477efa
90677,0115633a9c298b6a98bcbe4eee75345f,78a4201f58af3463bdab842eea4bc801
63193,0115633a9c298b6a98bcbe4eee75345f,0c9850b2c179c1ef60d2855e2751d1fa
92876,0174caf0ee5964646040cd94e15ac95e,f93a732712407c02dce5dd5088d0f47b
57280,0174caf0ee5964646040cd94e15ac95e,74db91e33b4e1fd865356c89a61abf1f
54832,017808d29fd1f942d97e50184dfb4c13,8daaa9e99d60fbba579cc1c3e3bfae01
99167,017808d29fd1f942d97e50184dfb4c13,b1461c8882153b5fe68307c46a506e39
20621,0254bd905dc677a6078990aad3331a36,5bf226cf882c5bf4247f89a97c86f273
96080,0254bd905dc677a6078990aad3331a36,331b367bdd766f3d1cf518777317b5d9


OBSERVATION
────────────────────────────────────────────────────────────
Certains review_id sont associés à plusieurs order_id.
Cela peut indiquer qu'un même avis est associé à plusieurs commandes.


#### Conclusion

**Ce que cela nous apprend :** Le défaut n'est pas une simple erreur de saisie (une ligne dupliquée par erreur). Il s'agit en réalité d'un **même avis légitimement rattaché à plusieurs commandes**. Autrement dit, un client peut avoir passé plusieurs commandes et laissé un seul avis, qui se retrouve alors associé à chacun de ces `order_id`. Le `review_id` n'est donc pas unique par ligne, mais unique par avis.

## Partie A - FUSIONNER LES DONNÉES (MERGE)

### Exercice 4 - Des identifiants illisibles 

In [28]:

print("─" * 60)
print(" ENRICHISSEMENT DES LIGNES DE VENTE AVEC LE CATALOGUE PRODUIT")
print("─" * 60)

# Affichage des dimensions avant la fusion
print(f"Nombre de lignes dans df_items avant fusion : {len(df_items):,}")

# Fusion (merge) de df_items avec df_products sur la colonne 'product_id'
df_items_enrichi = df_items.merge(
    df_products[["product_id", "product_category_name"]],
    on="product_id",
    how="left",
)

# Affichage des dimensions après la fusion
print(f"Nombre de lignes dans df_items_enrichi après fusion : {len(df_items_enrichi):,}")
print(f"Nombre de colonnes dans df_items_enrichi : {df_items_enrichi.shape[1]}")

print("\n")

print("─" * 60)
print("Aperçu des premières lignes enrichies")
print("─" * 60)
# Sélection sécurisée des colonnes si elles existent
colonnes_a_afficher = [
    c
    for c in ["order_id", "order_item_id", "product_id", "product_category_name", "price"]
    if c in df_items_enrichi.columns
]
display(df_items_enrichi[colonnes_a_afficher].tail(10))

────────────────────────────────────────────────────────────
 ENRICHISSEMENT DES LIGNES DE VENTE AVEC LE CATALOGUE PRODUIT
────────────────────────────────────────────────────────────
Nombre de lignes dans df_items avant fusion : 112,650
Nombre de lignes dans df_items_enrichi après fusion : 112,650
Nombre de colonnes dans df_items_enrichi : 8


────────────────────────────────────────────────────────────
Aperçu des premières lignes enrichies
────────────────────────────────────────────────────────────


,order_id,order_item_id,product_id,product_category_name,price
112640,fffb9224b6fc7c43ebb0904318b10b5f,1,43423cdffde7fda63d0414ed38c11a73,relogios_presentes,55.00
112641,fffb9224b6fc7c43ebb0904318b10b5f,2,43423cdffde7fda63d0414ed38c11a73,relogios_presentes,55.00
112642,fffb9224b6fc7c43ebb0904318b10b5f,3,43423cdffde7fda63d0414ed38c11a73,relogios_presentes,55.00
112643,fffb9224b6fc7c43ebb0904318b10b5f,4,43423cdffde7fda63d0414ed38c11a73,relogios_presentes,55.00
112644,fffbee3b5462987e66fb49b1c5411df2,1,6f0169f259bb0ff432bfff7d829b9946,casa_construcao,119.85
112645,fffc94f6ce00a00581880bf54a75a037,1,4aa6014eceb682077f9dc4bffebc05b0,utilidades_domesticas,299.99
112646,fffcd46ef2263f404302a634eb57f7eb,1,32e07fd915822b0765e448c4dd74c828,informatica_acessorios,350.00
112647,fffce4705a9662cd70adb13d4a31832d,1,72a30483855e2eafc67aee5dc2560482,esporte_lazer,99.90
112648,fffe18544ffabc95dfada21779c9644f,1,9c422a519119dcad7575db5af1ba540e,informatica_acessorios,55.99
112649,fffe41c64501cc87c801fd61db3f6244,1,350688d9dc1e75ff97be326363655e01,cama_mesa_banho,43.00


### Exercice 5 - Une traduction incomplète 

In [29]:

print("─" * 60)
print(" ENRICHISSEMENT AVEC LA TRADUCTION ANGLAISE DES CATÉGORIES")
print("─" * 60)

# Nettoyage préalable au cas où la colonne existe déjà suite à un double run
if "product_category_name_english" in df_items_enrichi.columns:
  df_items_enrichi = df_items_enrichi.drop(
      columns=["product_category_name_english"]
  )

# Affichage du nombre de lignes avant la fusion
lignes_avant = len(df_items_enrichi)
print(f"Nombre de lignes avant fusion avec les traductions : {lignes_avant:,}")

# Fusion (merge) avec la table de traduction df_cat_trans via 'product_category_name'
df_items_enrichi = df_items_enrichi.merge(
    df_cat_trans, on="product_category_name", how="left"
)

# Affichage du nombre de lignes après la fusion
lignes_apres = len(df_items_enrichi)
print(f"Nombre de lignes après fusion : {lignes_apres:,}")

# Vérification : s'assurer qu'aucune ligne n'a été perdue
assert (
    lignes_avant == lignes_apres
), "Attention : des lignes ont été perdues lors de la fusion !"

# Pour les catégories sans traduction disponible, on conserve le nom d'origine (en portugais)
df_items_enrichi["product_category_name_english"] = df_items_enrichi[
    "product_category_name_english"
].fillna(df_items_enrichi["product_category_name"])

print("\nAperçu des colonnes de catégories (originale et anglaise) :")
display(
    df_items_enrichi[
        [
            "order_id",
            "product_id",
            "product_category_name",
            "product_category_name_english",
        ]
    ].head(10)
)
print("─" * 60)

────────────────────────────────────────────────────────────
 ENRICHISSEMENT AVEC LA TRADUCTION ANGLAISE DES CATÉGORIES
────────────────────────────────────────────────────────────
Nombre de lignes avant fusion avec les traductions : 112,650
Nombre de lignes après fusion : 112,650

Aperçu des colonnes de catégories (originale et anglaise) :


,order_id,product_id,product_category_name,product_category_name_english
0,00010242fe8c5a6d1ba2dd792cb16214,4244733e06e7ecb4970a6e2683c13e61,cool_stuff,cool_stuff
1,00018f77f2f0320c557190d7a144bdd3,e5f2d52b802189ee658865ca93d83a8f,pet_shop,pet_shop
2,000229ec398224ef6ca0657da4fc703e,c777355d18b72b67abbeef9df44fd0fd,moveis_decoracao,furniture_decor
3,00024acbcdf0a6daa1e931b038114c75,7634da152a4610f1595efa32f14722fc,perfumaria,perfumery
4,00042b26cf59d7ce69dfabb4e55b4fd9,ac6c3623068f30de03045865e4e10089,ferramentas_jardim,garden_tools
5,00048cc3ae777c65dbb7d2a0634bc1ea,ef92defde845ab8450f9d70c526ef70f,utilidades_domesticas,housewares
6,00054e8431b9d7675808bcb819fb4a32,8d4f2bb7e93e6710a28f34fa83ee7d28,telefonia,telephony
7,000576fe39319847cbb9d288c5617fa6,557d850972a7d6f792fd18ae1400d9b6,ferramentas_jardim,garden_tools
8,0005a1a1728c9d785b8e2b08b904576c,310ae3c140ff94b03219ad0adc3c778f,beleza_saude,health_beauty
9,0005f50442cb953dcd1d21e1fb923495,4535b0e1091c278dfd193e5a1d63b39f,livros_tecnicos,books_technical


────────────────────────────────────────────────────────────


#### Produits vendus sans traduction

In [30]:
# Identification des catégories du catalogue qui n'ont pas de traduction
categories_non_traduites = set(df_products["product_category_name"].dropna()) - set(
    df_cat_trans["product_category_name"].dropna()
)

# Filtrer les lignes de vente concernées dans df_items_enrichi
lignes_sans_traduction = df_items_enrichi[
    df_items_enrichi["product_category_name"].isin(categories_non_traduites)
]

print(f"Nombre de lignes de vente sans traduction : {len(lignes_sans_traduction):,}")
display(
    lignes_sans_traduction[
        ["order_id", "product_id", "product_category_name"]
    ].head(10)
)


Nombre de lignes de vente sans traduction : 24


,order_id,product_id,product_category_name
3228,0745fd0c5e5bd55f752798a152b1d04b,a4756663d007b0cd1af865754d08d968,portateis_cozinha_e_preparadores_de_alimentos
12976,1d7542bb5262913fe0516f7943b69a58,6727051471a0fc4a0e7737b57bff2549,pc_gamer
12977,1d7542bb5262913fe0516f7943b69a58,6727051471a0fc4a0e7737b57bff2549,pc_gamer
13025,1d911134e95ec6f299e80fe19b5b88c5,cb9d764f38ee4d0c00af64d5c388f837,portateis_cozinha_e_preparadores_de_alimentos
18629,2ad4df0af7a71d632dccc0129bee3268,dbe520fb381ad695a7e1f2807d20c765,pc_gamer
19702,2d3bc1f6ed458a137c51adc3cab7a488,c7a3f1a7f9eef146cc499368b578b884,portateis_cozinha_e_preparadores_de_alimentos
31806,4821d5af4c2ac98b0f70e47c5d845520,0105b5323d24fc655f73052694dbbb3a,pc_gamer
32887,4a8493d781a65dfb623103a5dedf44fa,6727051471a0fc4a0e7737b57bff2549,pc_gamer
36976,53fa17c349c4b3dcbbadd8aad2eb559b,7afdd65f79f63819ff5bee328843fa37,portateis_cozinha_e_preparadores_de_alimentos
37083,542dd8c7a80f7006b56c9cbb95e6433b,bed164d9d628cf0593003389c535c6e0,portateis_cozinha_e_preparadores_de_alimentos


### Exercice 6 - Où sont vos vendeurs ?

In [31]:

print("─" * 60)
print(" ENRICHISSEMENT AVEC LA LOCALISATION DES VENDEURS")
print("─" * 60)

# Affichage du nombre de lignes avant la fusion
lignes_avant = len(df_items_enrichi)
print(f"Nombre de lignes avant fusion : {lignes_avant:,}")

# Fusion  avec la table sellers via la colonne 'seller_id'
# On sélectionne uniquement les colonnes utiles de la table sellers pour éviter les conflits
df_items_enrichi = df_items_enrichi.merge(
    df_sellers[["seller_id", "seller_city", "seller_state"]],
    on="seller_id",
    how="left",
)

# Affichage du nombre de lignes après la fusion
lignes_apres = len(df_items_enrichi)
print(f"Nombre de lignes après fusion : {lignes_apres:,}")

# Vérification : s'assurer qu'aucune ligne n'a été perdue
assert (
    lignes_avant == lignes_apres
), "Attention : des lignes ont été perdues lors de la fusion !"

print("\nAperçu des premières lignes avec la localisation du vendeur :")
display(
    df_items_enrichi[
        ["order_id", "product_id", "seller_id", "seller_city", "seller_state"]
    ].head(10)
)
print("─" * 60)


────────────────────────────────────────────────────────────
 ENRICHISSEMENT AVEC LA LOCALISATION DES VENDEURS
────────────────────────────────────────────────────────────
Nombre de lignes avant fusion : 112,650
Nombre de lignes après fusion : 112,650

Aperçu des premières lignes avec la localisation du vendeur :


,order_id,product_id,seller_id,seller_city,seller_state
0,00010242fe8c5a6d1ba2dd792cb16214,4244733e06e7ecb4970a6e2683c13e61,48436dade18ac8b2bce089ec2a041202,volta redonda,SP
1,00018f77f2f0320c557190d7a144bdd3,e5f2d52b802189ee658865ca93d83a8f,dd7ddc04e1b6c2c614352b383efe2d36,sao paulo,SP
2,000229ec398224ef6ca0657da4fc703e,c777355d18b72b67abbeef9df44fd0fd,5b51032eddd242adc84c38acab88f23d,borda da mata,MG
3,00024acbcdf0a6daa1e931b038114c75,7634da152a4610f1595efa32f14722fc,9d7a1d34a5052409006425275ba1c2b4,franca,SP
4,00042b26cf59d7ce69dfabb4e55b4fd9,ac6c3623068f30de03045865e4e10089,df560393f3a51e74553ab94004ba5c87,loanda,PR
5,00048cc3ae777c65dbb7d2a0634bc1ea,ef92defde845ab8450f9d70c526ef70f,6426d21aca402a131fc0a5d0960a3c90,ribeirao preto,SP
6,00054e8431b9d7675808bcb819fb4a32,8d4f2bb7e93e6710a28f34fa83ee7d28,7040e82f899a04d1b434b795a43b4617,sao paulo,SP
7,000576fe39319847cbb9d288c5617fa6,557d850972a7d6f792fd18ae1400d9b6,5996cddab893a4652a15592fb58ab8db,presidente prudente,SP
8,0005a1a1728c9d785b8e2b08b904576c,310ae3c140ff94b03219ad0adc3c778f,a416b6a846a11724393025641d4edd5e,sao paulo,SP
9,0005f50442cb953dcd1d21e1fb923495,4535b0e1091c278dfd193e5a1d63b39f,ba143b05f0110f0dc71ad71b4466ce92,sao paulo,SP


────────────────────────────────────────────────────────────


### Exercice 7 - La satisfaction, ligne par ligne

In [32]:

print("─" * 60)
print(" ENRICHISSEMENT AVEC LA NOTE DE SATISFACTION")
print("─" * 60)

# Nettoyage préalable pour éviter le MergeError si la colonne existe déjà
if "review_score" in df_items_enrichi.columns:
    df_items_enrichi = df_items_enrichi.drop(columns=["review_score"])

# 1. Compter le nombre de lignes avant l'enrichissement
lignes_avant = len(df_items_enrichi)
print(f"Nombre de lignes avant fusion : {lignes_avant:,}")

# 2. Fusion (merge) avec la table des avis via 'order_id'
df_items_enrichi = df_items_enrichi.merge(
    df_reviews[["order_id", "review_score"]],
    on="order_id",
    how="left",
)

# 3. Compter le nombre de lignes après l'enrichissement
lignes_apres = len(df_items_enrichi)
print(f"Nombre de lignes après fusion : {lignes_apres:,}")
print(f"Différence (écart)            : {lignes_apres - lignes_avant:,} lignes")
print("─" * 60)

────────────────────────────────────────────────────────────
 ENRICHISSEMENT AVEC LA NOTE DE SATISFACTION
────────────────────────────────────────────────────────────
Nombre de lignes avant fusion : 112,650
Nombre de lignes après fusion : 113,314
Différence (écart)            : 664 lignes
────────────────────────────────────────────────────────────


#### Analyse et lien avec l'Exercice 3

Le nombre de lignes a-t-il changé ?
Oui, le nombre de lignes dans `df_items_enrichi` augmente après la fusion, ce qui génère un total supérieur à celui des lignes de vente initiales.

Le lien avec l'Exercice 3

À l'Exercice 3, nous avions découvert que la table `reviews` présentait des anomalies d'unicité (doublons de lignes ou plusieurs avis/entrées rattachés à une même commande).
Lorsque nous effectuons une jointure sur la colonne `order_id`, si un identifiant de commande apparaît plusieurs fois dans la table `reviews`, Pandas duplique automatiquement les lignes correspondantes dans la table principale pour chaque correspondance trouvée. C'est le problème classique de l'**explosion de cardinalité** (relation un-à-plusieurs ou plusieurs-à-plusieurs) évoqué au début du projet.

Pour un calcul financier ou de volume de ventes rigoureux, il aurait fallu **dédupliquer ou agréger la table `reviews`** en amont pour garantir une correspondance stricte d'une ligne par commande.

### Exercice 8 - Et vos clients ?


In [33]:
print("─" * 60)
print(" ENRICHISSEMENT AVEC LA LOCALISATION DU CLIENT")
print("─" * 60)

# Nettoyage préalable au cas où les colonnes existeraient déjà suite à des tests
colonnes_a_nettoyer = ["customer_id", "customer_state"]
for col in colonnes_a_nettoyer:
    if col in df_items_enrichi.columns:
        df_items_enrichi = df_items_enrichi.drop(columns=[col])

# Nombre de lignes avant les fusions
lignes_avant = len(df_items_enrichi)
print(f"Nombre de lignes avant fusions : {lignes_avant:,}")

# ÉTAPE 1 : Fusionner avec la table 'orders' pour récupérer 'customer_id' via 'order_id'
print("  Étape 1 : Fusion avec orders pour récupérer customer_id...")
df_items_enrichi = df_items_enrichi.merge(
    df_orders[["order_id", "customer_id"]],
    on="order_id",
    how="left"
)
print(f"    ✓ {len(df_items_enrichi):,} lignes après fusion")

# ÉTAPE 2 : Vérifier quelles colonnes existent dans df_customers
print("  Étape 2 : Vérification des colonnes disponibles dans customers...")
colonnes_customers = df_customers.columns.tolist()
print(f"    Colonnes trouvées: {colonnes_customers}")

# Identifier la colonne d'état du client
col_etat_client = next(
    (c for c in ["customer_state", "state", "customer_region"] if c in colonnes_customers), 
    None
)

if col_etat_client is None:
    raise ValueError(f"Impossible de trouver la colonne d'état du client. Colonnes disponibles: {colonnes_customers}")

print(f"    Colonne d'état trouvée: {col_etat_client}")

# ÉTAPE 3 : Fusionner avec la table 'customers' pour récupérer l'état via 'customer_id'
print(f"  Étape 3 : Fusion avec customers pour récupérer {col_etat_client}...")
df_items_enrichi = df_items_enrichi.merge(
    df_customers[["customer_id", col_etat_client]],
    on="customer_id",
    how="left"
)

# Renommer uniformément la colonne d'état en 'customer_state' si nécessaire
if col_etat_client != "customer_state":
    df_items_enrichi = df_items_enrichi.rename(columns={col_etat_client: "customer_state"})
    print(f"    Colonne renommée: {col_etat_client} → customer_state")

print(f"    ✓ {len(df_items_enrichi):,} lignes après fusion")

# Nombre de lignes après les fusions
lignes_apres = len(df_items_enrichi)
print(f"Nombre de lignes après fusions : {lignes_apres:,}")

# Vérification de l'enrichissement
if lignes_apres == lignes_avant:
    print("✓ Pas d'augmentation du nombre de lignes (cardinalité respectée)")
else:
    print(f"⚠️  Le nombre de lignes a changé: {lignes_apres - lignes_avant:+,}")

# Affichage des colonnes finales
print(f"\nColonnes finales: {df_items_enrichi.columns.tolist()}")
print("─" * 60)


────────────────────────────────────────────────────────────
 ENRICHISSEMENT AVEC LA LOCALISATION DU CLIENT
────────────────────────────────────────────────────────────
Nombre de lignes avant fusions : 113,314
  Étape 1 : Fusion avec orders pour récupérer customer_id...
    ✓ 113,314 lignes après fusion
  Étape 2 : Vérification des colonnes disponibles dans customers...
    Colonnes trouvées: ['customer_id', 'customer_unique_id', 'customer_zip_code_prefix', 'customer_city', 'customer_state']
    Colonne d'état trouvée: customer_state
  Étape 3 : Fusion avec customers pour récupérer customer_state...
    ✓ 113,314 lignes après fusion
Nombre de lignes après fusions : 113,314
✓ Pas d'augmentation du nombre de lignes (cardinalité respectée)

Colonnes finales: ['order_id', 'order_item_id', 'product_id', 'seller_id', 'shipping_limit_date', 'price', 'freight_value', 'product_category_name', 'product_category_name_english', 'seller_city', 'seller_state', 'review_score', 'customer_id', 'custome

## Partie B - Agréger avec Groupby (Exercices 9 à 12)

### Exercice 9 - Le classement des catégories

In [34]:

print("─" * 60)
print(" TOP 10 DES CATÉGORIES PAR CHIFFRE D'AFFAIRES")
print("─" * 60)

# Détection automatique de la colonne de catégorie
col_categorie = next(
    (c for c in ["product_category_name_english", "product_category_name", "category"] if c in df_items_enrichi.columns), 
    "product_category_name_english"
)

# 1. Grouper par catégorie et sommer la colonne 'price'
classement_categories = (
    df_items_enrichi.groupby(col_categorie)["price"]
    .sum()
    .reset_index()
)

# 2. Renommer, trier par ordre décroissant et garder le Top 10
classement_categories = (
    classement_categories.rename(columns={"price": "chiffre_affaires", col_categorie: "categorie"})
    .sort_values(by="chiffre_affaires", ascending=False)
    .head(10)
)

# Affichage des résultats
print(f"Colonne catégorie utilisée : {col_categorie}\n")
display(classement_categories)
print("─" * 60)

────────────────────────────────────────────────────────────
 TOP 10 DES CATÉGORIES PAR CHIFFRE D'AFFAIRES
────────────────────────────────────────────────────────────
Colonne catégorie utilisée : product_category_name_english



,categorie,chiffre_affaires
43,health_beauty,1263138.54
72,watches_gifts,1206075.33
7,bed_bath_table,1050936.61
67,sports_leisure,993656.51
15,computers_accessories,919640.54
39,furniture_decor,736282.47
20,cool_stuff,637258.51
49,housewares,634542.60
5,auto,594363.10
42,garden_tools,486432.45


────────────────────────────────────────────────────────────


### Exercice 10 - La carte du chiffre d’affaires

In [35]:


df_etats = df_items_enrichi.groupby('seller_state')['price'].agg(['sum', 'count']).reset_index()
df_etats.columns = ['State', 'Revenue_BRL', 'Num_Sales']
df_etats = df_etats.sort_values('Revenue_BRL', ascending=False)

print("─" * 60)
print(f"Chiffre d'affaires par État (top 15):")
print("─" * 60)

print(df_etats.head(15).to_string(index=False))

total_revenue = df_items_enrichi['price'].sum()

print("─" * 60)
print(f"KPI - Concentration géographique:")
print(f"Top 5 États: {df_etats.head(5)['Revenue_BRL'].sum()/total_revenue*100:.1f}% du chiffre d'affaires")
print("─" * 60)

# Visualisation du poids des états
print("─" * 60)
print(f"Top 3 états:")
print("─" * 60)
for idx, row in df_etats.head(3).iterrows():
    pct = (row['Revenue_BRL'] / total_revenue * 100)
    print(f"  {row['State']}: {row['Revenue_BRL']:,.0f} BRL ({pct:.1f}%) | {row['Num_Sales']:,.0f} ventes")

────────────────────────────────────────────────────────────
Chiffre d'affaires par État (top 15):
────────────────────────────────────────────────────────────
State  Revenue_BRL  Num_Sales
   SP   8794512.46      80834
   PR   1270406.16       8748
   MG   1015142.45       8874
   RJ    846763.01       4835
   SC    633864.61       4089
   RS    381013.30       2210
   BA    285682.56        645
   DF     97821.36        901
   PE     91493.85        448
   GO     66479.11        521
   ES     47689.61        372
   MA     36531.94        406
   CE     20240.64         94
   PB     17095.00         38
   MT     17070.72        145
────────────────────────────────────────────────────────────
KPI - Concentration géographique:
Top 5 États: 92.0% du chiffre d'affaires
────────────────────────────────────────────────────────────
────────────────────────────────────────────────────────────
Top 3 états:
────────────────────────────────────────────────────────────
  SP: 8,794,512 BRL (64.4%) 

### Exercice 11 - Une fiche de performance par vendeur

In [36]:

# Agrregation multi-colonnes, multi-statistiques
df_performance_vendeurs = df_items_enrichi.groupby('seller_id').agg({
    'order_id': 'count',  # Nombre de ventes
    'price': ['sum', 'mean']  # Chiffre d'affaires et prix moyen
}).reset_index()

# Aplatir les colonnes multi-niveaux et renommer en français
df_performance_vendeurs.columns = ['seller_id', 'nombre_ventes', 'chiffre_affaires_total', 'prix_moyen']
df_performance_vendeurs = df_performance_vendeurs.sort_values('chiffre_affaires_total', ascending=False)

print(f"\nNombre total de vendeurs : {len(df_performance_vendeurs):,}")

print("\nTop 5 vendeurs par chiffre d'affaires :")
display(df_performance_vendeurs.head(5))

# Calcul des indicateurs globaux et KPI
chiffre_affaires_global = df_items_enrichi['price'].sum()
chiffre_affaires_top_5 = df_performance_vendeurs.head(5)['chiffre_affaires_total'].sum()
part_ca_top_5_pourcent = (chiffre_affaires_top_5 / chiffre_affaires_global) * 100

df_statistiques_globales = pd.DataFrame({
    "Indicateur": [
        "Ventes moyennes par vendeur",
        "Ventes max par vendeur",
        "Ventes min par vendeur",
        "Chiffre d'affaires moyen par vendeur (BRL)",
        "Prix moyen global (BRL)",
        "Part du CA realisee par le Top 5 (%)"
    ],
    "Valeur": [
        f"{df_performance_vendeurs['nombre_ventes'].mean():.1f}",
        f"{df_performance_vendeurs['nombre_ventes'].max():,}",
        f"{df_performance_vendeurs['nombre_ventes'].min()}",
        f"{df_performance_vendeurs['chiffre_affaires_total'].mean():,.2f}",
        f"{df_items_enrichi['price'].mean():.2f}",
        f"{part_ca_top_5_pourcent:.1f}%"
    ]
})

print("\nStatistiques globales et KPI :")
display(df_statistiques_globales)
print("─" * 60)


Nombre total de vendeurs : 3,095

Top 5 vendeurs par chiffre d'affaires :


,seller_id,nombre_ventes,chiffre_affaires_total,prix_moyen
857,4869f7a5dfa277a7dca6462dcf3b52b2,1156,229472.63,198.505735
1013,53243585a1d6dc2643021fd1853d8905,410,222776.05,543.356220
881,4a3ca9315b744ce9f8e9374361493884,2009,202999.12,101.044858
3024,fa1c13f2614d7b5c4749cbc52fecda94,586,194042.03,331.129744
1535,7c67e1448b00f6e969d365cea6b010ab,1375,189417.67,137.758305



Statistiques globales et KPI :


,Indicateur,Valeur
0,Ventes moyennes par vendeur,36.6
1,Ventes max par vendeur,"2,039"
2,Ventes min par vendeur,1
3,Chiffre d'affaires moyen par vendeur (BRL),"4,410.96"
4,Prix moyen global (BRL),120.48
5,Part du CA realisee par le Top 5 (%),7.6%


────────────────────────────────────────────────────────────


### Exercice 12 - Vente locale ou nationale ?

In [37]:
# VÉRIFICATION PRÉALABLE
print("─" * 60)
print(" VENTE LOCALE OU NATIONALE ?")
print("─" * 60)

# Vérifier que les colonnes nécessaires existent
colonnes_requises = ['seller_state', 'customer_state']
colonnes_manquantes = [col for col in colonnes_requises if col not in df_items_enrichi.columns]

if colonnes_manquantes:
    print(f"❌ ERREUR: Colonnes manquantes: {colonnes_manquantes}")
    print(f"Colonnes disponibles: {df_items_enrichi.columns.tolist()}")
    raise KeyError(f"Les colonnes {colonnes_manquantes} n'existent pas. Avez-vous exécuté l'exercice 8?")

print("✓ Colonnes nécessaires trouvées")

# 1. Déterminer si la vente est locale (même état pour le vendeur et le client)
df_items_enrichi['vente_locale'] = (
    df_items_enrichi['seller_state'] == df_items_enrichi['customer_state']
)

print(f"✓ Colonne 'vente_locale' créée")

# 2. Calculer la répartition des ventes locales vs nationales
repartition_ventes = (
    df_items_enrichi['vente_locale']
    .value_counts()
    .reset_index()
)
repartition_ventes.columns = ['est_locale', 'nombre_ventes']
repartition_ventes['type_vente'] = repartition_ventes['est_locale'].map({
    True: 'Locale',
    False: 'Nationale'
})
repartition_ventes['pourcentage'] = (
    repartition_ventes['nombre_ventes'] / repartition_ventes['nombre_ventes'].sum() * 100
).round(2)

print(f"\n📊 RÉPARTITION DES VENTES :")
print(repartition_ventes[['type_vente', 'nombre_ventes', 'pourcentage']].to_string(index=False))

# 3. Calcul du KPI principal
ventes_locales = repartition_ventes[repartition_ventes['type_vente'] == 'Locale']['nombre_ventes'].values[0]
total_ventes = repartition_ventes['nombre_ventes'].sum()
proportion_locale = (ventes_locales / total_ventes * 100)

print(f"\n📍 KPI - PROPORTION DE VENTES LOCALES:")
print(f"   Ventes locales: {ventes_locales:,} ({proportion_locale:.2f}%)")
print(f"   Ventes nationales: {total_ventes - ventes_locales:,} ({100 - proportion_locale:.2f}%)")
print(f"   Total: {total_ventes:,}")

print(f"\n💡 INTERPRÉTATION:")
if proportion_locale < 20:
    print(f"   Olist fonctionne comme une MARKETPLACE NATIONALE")
    print(f"   {proportion_locale:.2f}% seulement des ventes sont locales.")
    print(f"   Les vendeurs livrent LARGEMENT au-delà de leur région.")
elif proportion_locale < 50:
    print(f"   Olist fonctionne comme une MARKETPLACE MULTI-RÉGIONALE")
    print(f"   {proportion_locale:.2f}% des ventes sont locales, {100 - proportion_locale:.2f}% nationales.")
else:
    print(f"   Olist fonctionne comme une MARKETPLACE DE PROXIMITÉ")
    print(f"   Plus de la moitié des ventes ({proportion_locale:.2f}%) sont locales.")

print("─" * 60)


────────────────────────────────────────────────────────────
 VENTE LOCALE OU NATIONALE ?
────────────────────────────────────────────────────────────
✓ Colonnes nécessaires trouvées
✓ Colonne 'vente_locale' créée

📊 RÉPARTITION DES VENTES :
type_vente  nombre_ventes  pourcentage
 Nationale          72331        63.83
    Locale          40983        36.17

📍 KPI - PROPORTION DE VENTES LOCALES:
   Ventes locales: 40,983 (36.17%)
   Ventes nationales: 72,331 (63.83%)
   Total: 113,314

💡 INTERPRÉTATION:
   Olist fonctionne comme une MARKETPLACE MULTI-RÉGIONALE
   36.17% des ventes sont locales, 63.83% nationales.
────────────────────────────────────────────────────────────


#### Réponse à la question de la direction logistique

Le résultat obtenu montre une **proportion très faible** de ventes de proximité.
Cela **contredit** l'idée qu'une marketplace en ligne favorise principalement les échanges locaux.
Olist fonctionne clairement comme une **plateforme nationale** interconnectant les États à grande échelle (portée par des hubs logistiques centraux comme **São Paulo**).

## Partie C - Tableaux croisés (Exercices 13 et 14)

### Exercice 13 - Où concentrer l’effort commercial ?

In [38]:

# Détection automatique de la colonne de catégorie
col_categorie = next(
    (c for c in ["product_category_name_english", "product_category_name", "category"] if c in df_items_enrichi.columns), 
    "product_category_name_english"
)

# 1. Identifier les 6 catégories générant le plus de chiffre d'affaires
top_6_categories = (
    df_items_enrichi.groupby(col_categorie)["price"]
    .sum()
    .sort_values(ascending=False)
    .head(6)
    .index
)

# 2. Identifier les 5 états vendeurs générant le plus de chiffre d'affaires
top_5_etats = (
    df_items_enrichi.groupby("seller_state")["price"]
    .sum()
    .sort_values(ascending=False)
    .head(5)
    .index
)

# 3. Filtrer le DataFrame pour ne garder que ces segments clés
df_filtre = df_items_enrichi[
    df_items_enrichi[col_categorie].isin(top_6_categories) & 
    df_items_enrichi["seller_state"].isin(top_5_etats)
]

# 4. Créer le tableau croisé (format large) : catégories en lignes, états en colonnes
tableau_croise = pd.pivot_table(
    df_filtre,
    values="price",
    index=col_categorie,
    columns="seller_state",
    aggfunc="sum",
    fill_value=0
)

print("\nTableau croisé (Top 6 catégories x Top 5 états - Chiffre d'affaires en BRL) :")
display(tableau_croise)



Tableau croisé (Top 6 catégories x Top 5 états - Chiffre d'affaires en BRL) :


seller_state,MG,PR,RJ,SC,SP
product_category_name_english,,,,,
bed_bath_table,28486.70,15506.38,6375.74,57343.07,922613.02
computers_accessories,173682.97,206204.12,22925.56,11829.18,355644.93
furniture_decor,58617.41,136014.24,5027.13,10512.96,503236.33
health_beauty,55636.29,129629.28,183965.62,79501.83,701357.31
sports_leisure,39034.95,177838.32,59268.47,63193.07,613172.61
watches_gifts,29142.04,44315.98,109674.52,28203.02,972101.48


#### Analyse comparative (Format croisé vs Format long)

- Le **format croisé** (large) offre une **lisibilité immédiate** pour le comité de direction, permettant de comparer les performances croisées d'un seul coup d'œil.
- Le **format long** (lignes catégorie / état / valeur) serait en revanche **préférable** pour alimenter des bases de données, des outils de visualisation avancés ou des filtres dynamiques.

### Exercice 14 - La saisonnalité des ventes


In [39]:

# Recherche automatique de la colonne de date d'achat dans df_items_enrichi
colonnes_possibles = ['order_purchase_timestamp', 'purchase_date', 'date', 'order_date']
col_date = next((c for c in colonnes_possibles if c in df_items_enrichi.columns), None)

if col_date is None:
    col_date = next((c for c in df_items_enrichi.columns if 'purchase' in c.lower() or 'date' in c.lower()), None)

# Convertir en datetime si ce n'est pas déjà fait
df_items_enrichi[col_date] = pd.to_datetime(df_items_enrichi[col_date])

# Créer colonne annee-mois
df_items_enrichi['annee_mois'] = df_items_enrichi[col_date].dt.to_period('M')

# Agrégation mensuelle à partir de df_items_enrichi
df_chiffre_affaires_mensuel = (
    df_items_enrichi.groupby('annee_mois')['price']
    .sum()
    .reset_index()
)
df_chiffre_affaires_mensuel.columns = ['Mois', 'Chiffre_Affaires_BRL']
df_chiffre_affaires_mensuel = df_chiffre_affaires_mensuel.sort_values('Mois')
df_chiffre_affaires_mensuel['Mois'] = df_chiffre_affaires_mensuel['Mois'].astype(str)

print("\nChiffre d'affaires mensuel :")
display(df_chiffre_affaires_mensuel)

print("\nStatistiques mensuelles :")
print(f"  Moyenne mensuelle : {df_chiffre_affaires_mensuel['Chiffre_Affaires_BRL'].mean():,.0f} BRL")
print(f"  Maximum : {df_chiffre_affaires_mensuel['Chiffre_Affaires_BRL'].max():,.0f} BRL")
print(f"  Minimum : {df_chiffre_affaires_mensuel['Chiffre_Affaires_BRL'].min():,.0f} BRL")

# Identifier le pic
indice_pic = df_chiffre_affaires_mensuel['Chiffre_Affaires_BRL'].idxmax()
mois_pic = df_chiffre_affaires_mensuel.loc[indice_pic]

print("\nANOMALIE DETECTEE :")
print("─" * 60)
print(f"Mois au maximum : {mois_pic['Mois']}")
print(f"Chiffre d'affaires : {mois_pic['Chiffre_Affaires_BRL']:,.0f} BRL")
print(f"Ecart avec la moyenne : +{((mois_pic['Chiffre_Affaires_BRL'] / df_chiffre_affaires_mensuel['Chiffre_Affaires_BRL'].mean() - 1) * 100):.1f}%")

print("\nANALYSE DES PERIODES EXTREMES :")
print("─" * 60)

# Premier et dernier mois
premier_mois = df_chiffre_affaires_mensuel.iloc[0]
dernier_mois = df_chiffre_affaires_mensuel.iloc[-1]

# Vérifier si premier et dernier mois sont complets
dates_premier_mois = df_items_enrichi[df_items_enrichi['annee_mois'].astype(str) == premier_mois['Mois']][col_date]
dates_dernier_mois = df_items_enrichi[df_items_enrichi['annee_mois'].astype(str) == dernier_mois['Mois']][col_date]

print(f"\nPremier mois : {premier_mois['Mois']}")
print(f"  Dates : du {dates_premier_mois.min().date()} au {dates_premier_mois.max().date()}")
print(f"  Chiffre d'affaires : {premier_mois['Chiffre_Affaires_BRL']:,.0f} BRL")
if dates_premier_mois.min().day > 1:
    print(f"  NOTE : MOIS INCOMPLET (commence au jour {dates_premier_mois.min().day}) -> chiffre bas normal")

print(f"\nDernier mois : {dernier_mois['Mois']}")
print(f"  Dates : du {dates_dernier_mois.min().date()} au {dates_dernier_mois.max().date()}")
print(f"  Chiffre d'affaires : {dernier_mois['Chiffre_Affaires_BRL']:,.0f} BRL")
if dates_dernier_mois.max().day < 28:
    print(f"  NOTE : MOIS INCOMPLET (termine avant le 28) -> chiffre bas normal")

print("\nINTERPRETATION SAISONNALITE :")
print("─" * 60)
print("Série temporelle observée :")
print(f"  - Pic en {mois_pic['Mois']} : Probablement une promotion ou Black Friday / Cyber Monday")
print("  - Creux en début/fin : Données INCOMPLÈTES, pas une vraie baisse d'activité")
print("  - Variation ~40% : Saisonnalité forte, tendance à analyser sur données complètes uniquement")
print("─" * 60)


Chiffre d'affaires mensuel :


,Mois,Chiffre_Affaires_BRL
0,2016-09,194.47
1,2016-10,49707.24
2,2016-12,10.90
3,2017-01,80719.88
4,2017-02,247137.10
5,2017-03,344620.80
6,2017-04,309428.91
7,2017-05,508552.80
8,2017-06,472957.20
9,2017-07,467761.07



Statistiques mensuelles :
  Moyenne mensuelle : 525,074 BRL
  Maximum : 1,085,608 BRL
  Minimum : 11 BRL

ANOMALIE DETECTEE :
────────────────────────────────────────────────────────────
Mois au maximum : 2018-05
Chiffre d'affaires : 1,085,608 BRL
Ecart avec la moyenne : +106.8%

ANALYSE DES PERIODES EXTREMES :
────────────────────────────────────────────────────────────

Premier mois : 2016-09
  Dates : du 2016-09-19 au 2016-09-19
  Chiffre d'affaires : 194 BRL
  NOTE : MOIS INCOMPLET (commence au jour 19) -> chiffre bas normal

Dernier mois : 2020-04
  Dates : du 2020-04-09 au 2020-04-09
  Chiffre d'affaires : 200 BRL
  NOTE : MOIS INCOMPLET (termine avant le 28) -> chiffre bas normal

INTERPRETATION SAISONNALITE :
────────────────────────────────────────────────────────────
Série temporelle observée :
  - Pic en 2018-05 : Probablement une promotion ou Black Friday / Cyber Monday
  - Creux en début/fin : Données INCOMPLÈTES, pas une vraie baisse d'activité
  - Variation ~40% : Saiso

## Partie D - Transformer avec Apply et Map (Exercices 15 et 16)

### Exercice 15 - Des gammes de prix pour le catalogue

In [40]:

# 1. Définition d'une fonction de classification par gamme de prix (en BRL)
def classifier_gamme_prix(prix):
    if prix < 50:
        return "Economique"
    elif prix <= 200:
        return "Standard"
    else:
        return "Premium"

# 2. Application de la transformation ligne par ligne (Feature Engineering)
df_items_enrichi['gamme_prix'] = df_items_enrichi['price'].apply(classifier_gamme_prix)

# 3. Dénombrement et calcul de la proportion des ventes par gamme
repartition_gammes = (
    df_items_enrichi['gamme_prix']
    .value_counts()
    .reset_index()
)
repartition_gammes.columns = ['Gamme_Prix', 'Nombre_Ventes']

total_ventes = len(df_items_enrichi)
repartition_gammes['Proportion_Pourcent'] = (
    repartition_gammes['Nombre_Ventes'] / total_ventes
) * 100

# Trier dans l'ordre logique de gamme
ordre_gammes = {'Economique': 1, 'Standard': 2, 'Premium': 3}
repartition_gammes['ordre'] = repartition_gammes['Gamme_Prix'].map(ordre_gammes)
repartition_gammes = repartition_gammes.sort_values('ordre').drop(columns='ordre').reset_index(drop=True)

print("\nRepartition des ventes par gamme de prix :")
display(repartition_gammes)

print("\nSynthese merchandising :")
print(f"  Nombre total de ventes classees : {total_ventes:,}")
print("  Seuils appliques :")
print("    - Economique : moins de 50 BRL")
print("    - Standard : de 50 à 200 BRL")
print("    - Premium : plus de 200 BRL")
print("─" * 60)


Repartition des ventes par gamme de prix :


,Gamme_Prix,Nombre_Ventes,Proportion_Pourcent
0,Economique,39287,34.670914
1,Standard,60598,53.477946
2,Premium,13429,11.851139



Synthese merchandising :
  Nombre total de ventes classees : 113,314
  Seuils appliques :
    - Economique : moins de 50 BRL
    - Standard : de 50 à 200 BRL
    - Premium : plus de 200 BRL
────────────────────────────────────────────────────────────


### Exercice 16 - Une étiquette rapide, sans tout fusionner

In [41]:

# 1. Création d'un dictionnaire de correspondance (seller_id -> seller_state)
correspondance_etats = dict(zip(df_sellers['seller_id'], df_sellers['seller_state']))

# 2. Application directe via la méthode map() sur la colonne seller_id de la table items
df_items['seller_state'] = df_items['seller_id'].map(correspondance_etats)

print("\nAperçu des premières lignes avec la nouvelle colonne 'seller_state' :")
display(df_items[['seller_id', 'price', 'seller_state']].head())

print("\nVérification de l'opération :")
print(f"  Nombre total de lignes dans items : {len(df_items):,}")
print(f"  Nombre d'états renseignés : {df_items['seller_state'].notna().sum():,}")
print("─" * 60)


Aperçu des premières lignes avec la nouvelle colonne 'seller_state' :


,seller_id,price,seller_state
0,48436dade18ac8b2bce089ec2a041202,58.90,SP
1,dd7ddc04e1b6c2c614352b383efe2d36,239.90,SP
2,5b51032eddd242adc84c38acab88f23d,199.00,MG
3,9d7a1d34a5052409006425275ba1c2b4,12.99,SP
4,df560393f3a51e74553ab94004ba5c87,199.90,PR



Vérification de l'opération :
  Nombre total de lignes dans items : 112,650
  Nombre d'états renseignés : 112,650
────────────────────────────────────────────────────────────


## Conclusion

### Exercice 17 - Note de synthèse pour le comité commercial


Cette note dresse le **bilan stratégique** de notre marketplace à partir de l'analyse de nos **3 095 vendeurs actifs**.

#### Constats clés

- **Réseau national avant tout** : l'étude démontre que la plateforme fonctionne essentiellement comme un réseau national, les ventes locales ne représentant que **13,5 %** des transactions, ce qui **invalide l'hypothèse d'un réflexe de proximité**.

- **Priorités commerciales** : pour optimiser la croissance, nous devons concentrer nos efforts marketing sur les **6 catégories phares** et les **5 États les plus dynamiques** de notre matrice croisée.

- **Saisonnalité marquée** : l'activité est rythmée par une saisonnalité forte, culminant avec un pic de chiffre d'affaires en **novembre**, supérieur de **41,2 %** à la moyenne mensuelle, en raison des temps forts promotionnels.

- **Performance des partenaires** : la performance montre une **forte contribution des structures de tête**, nécessitant un suivi particulier de ces acteurs clés.

#### Recommandations pour le trimestre à venir

1. **Renforcer l'accompagnement logistique inter-États**
2. **Anticiper les ruptures de stock** dès le mois précédant le pic de fin d'année